# Char LM
이 노트북에서는 가장 기초적인 형태의 **decoder-only Transformer**를 구현합니다.

<img src="./img/charLM.png" width="800">

일반적인 서브워드 토큰 대신 각 글자(character)를 하나의 토큰으로 사용하는 character-level 방식을 사용하며, Transformer 기반 언어 모델의 기본 
구조와 학습 과정을 이해하는 것을 목표로 합니다.

## 1. text <-> token

### LLM의 학습 데이터

기본적인 **word-level tokenization**에서의 데이터 구성 방식은 다음과 같다.  
다만 실제 LLM에서는 보통 word-level보다 **subword tokenizer**를 사용한다.

예를 들어 다음 문장이 있다고 하자.

`나는 사과를 먹었다.`

Tokenizer가 이를 다음과 같이 변환한다고 가정하자.

| 토큰 |  | ID |
|---|:---:|---:|
| 나는 | → | 15 |
| 사과를 | → | 83 |
| 먹었다 | → | 233 |
| . | → | 3 |

그러면 문장은 다음과 같은 토큰 ID의 나열로 표현할 수 있다.


`tokens = [15, 83, 233, 3] `


LLM의 기본적인 학습 목표는 **이전 토큰들을 바탕으로 다음 토큰을 예측하는 것**이다.

따라서 입력과 정답을 한 칸씩 어긋나게 구성할 수 있다.


| 구분 | 토큰 |
|---|---|
| 입력 | [나는, 사과를, 먹었다] |
| 정답 |  [사과를, 먹었다, .] |

Causal Mask를 적용하면 하나의 시퀀스 안에서 다음과 같은 예측을 동시에 학습할 수 있다.

| 입력 |  | 정답 |
|---|:---:|---|
| 나는 | → | 사과를 |
| 나는 사과를 | → | 먹었다 |
| 나는 사과를 먹었다 | → | . |

즉, 토큰 시퀀스 `x`와 정답 시퀀스 `y`를 한 칸 차이 나게 구성하는 것만으로 **next-token prediction**을 위한 학습 데이터를 만들 수 있다.

한편 **character-level tokenization**에서는 단어 대신 문자 하나를 하나의 토큰으로 사용한다. 예를 들어 `사과`는 `사`, `과`라는 두 토큰으로 나뉠 수 있으며, 각각의 토큰은 모델 내부에서 하나의 정수형 **token ID**로 표현된다.

* input.txt는 한국어 위키백과의 인공지능 관련 문서에서 추출한 텍스트 3만자로로 구성되어 있습니다.

In [2]:
with open("input.txt", "r", encoding = "utf-8") as f:
    text = f.read()

print(text[:100])

인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 


### 전처리

위키백과의 위키문법을 제거하는 전처리를 진행한다.

In [3]:
import re

def preprocess(text):
    # 위키 문법 제거
    text = re.sub(r'=+', '', text)

    # HTML 공백
    text = text.replace('&#x20;', ' ')

    # 공백, 탭, 줄바꿈 ->  공백 1개
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

text = preprocess(text)

print(text[:100])
print("문자 수:", len(text))


인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 
문자 수: 29375


### 토크나이저 만들기

#### vocabulary 만들기

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(chars)
print(vocab_size)

[' ', '!', '"', '%', "'", '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '^', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '~', '·', '‘', '’', '“', '”', '《', '》', '人', '工', '智', '能', '가', '각', '간', '갈', '감', '갑', '갔', '강', '갖', '같', '개', '객', '거', '걱', '건', '걸', '검', '것', '게', '겠', '겨', '격', '겪', '견', '결', '겼', '경', '계', '고', '곤', '곧', '곳', '공', '과', '관', '광', '괴', '교', '구', '국', '군', '굴', '궁', '권', '귀', '규', '균', '그', '극', '근', '글', '금', '급', '기', '긴', '길', '깊', '까', '깎', '깔', '깨', '께', '껴', '꼈', '꼬', '꼽', '꽉', '꾸', '꿈', '끈', '끊', '끌', '끔', '끝', '나', '낙', '난', '날', '남', '났', '낮', '내', '낸', '낼', '냈', '냉', '너', '넌', '널', '넓', '넘', '넛', '넣', '네', '넷', '년', '념', '녔', '노', '논', '놀', '높', '놓', '뇌', '누', '눈', '눌', '뉴', '느', '는', '늘',

#### tokenizer 만들기

##### 파이썬 문법

##### 1. enumerate
```python
enumerate(iterable)
```

`iterable`: 리스트, 문자열, 튜플처럼 반복 가능한 것

enumerate(chars)는 기본적으로 `(인덱스, 값)` 쌍을 만들어줌.

In [49]:
'''
for item in enumerate(chars):
    print(item)
'''

'''
print(list(enumerate(chars)))
'''

for i,ch in enumerate(chars):
    if i<=10: 
        print(i, ch)
    elif i == 11: print(".\n.\n.")
    elif i>=750: print(i, ch)

0  
1 !
2 "
3 %
4 '
5 (
6 )
7 +
8 ,
9 -
10 .
.
.
.
750 흔
751 흥
752 희
753 히
754 힌
755 힐
756 힘


enumerate는 **반복 가능한 객체**를 만든다.

In [22]:
e = enumerate(chars)

print(next(e))

print(next(e))

print(next(e))

(0, ' ')
(1, '!')
(2, '"')


반복 가능한 객체의 각 원소에 인덱스를 붙이고, 반복할 때마다 다음 (인덱스, 값) 튜플을 내놓음

- 튜플 : 수정 불가 리스트

##### 2. dictionary comprehension 

```python
{key: value for item in iterable}
```

`(key, value)` 쌍의 딕셔너리를 만든다.


In [30]:
test_list = ['안', '녕', '하', '세', '요']
test_dictionary = {i : j for i, j in enumerate(test_list)}

print(test_list)
print(test_dictionary)

['안', '녕', '하', '세', '요']
{0: '안', 1: '녕', 2: '하', 3: '세', 4: '요'}


In [31]:
# 문자 -> 숫자
stoi = {ch: i for i, ch in enumerate(chars)}

# 숫자 -> 문자
itos = {i: ch for i, ch in enumerate(chars)}

딕셔너리는 key를 입력하면 그 key에 해당하는 value가 나온다.

In [32]:
sample_stoi = {
    'a' : 65,
    'b' : 67,
    'c' : 68
}

print(sample_stoi['a'])


65


In [40]:
# 인코더
def encode(s):
    return [stoi[c] for c in s]

# 디코더
def decode(tokens):
    return "".join(itos[i] for i in tokens)

In [43]:
tokens = encode("인공지능")

print(tokens)
print(decode(tokens))

[534, 121, 575, 200]
인공지능


## 2. Dataset && DataLoader